# Partial Differential Equations II

Chapter 11 of the TN23015 book: its explanation and its exercises. Run the first cell before anything else, then work down the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import time
from scipy.fftpack import dst,idst
from tn23015 import check_answers, given, load_data, md, print, row, show, slider

In this lecture, we will explore numerically solving partial differential equations (PDEs) that have the form of an initial value problem using the forward-time centered-space (FTCS) and the spectral method.

> **Learning goals**
>
> 1. Implement the FTCS method for solving initial-value PDEs
> 2. Solve and interpret the results from simulations of the 1-dimensional diffusion equation using FTCS
> 3. Identify the failure of the FTCS method for solving some problems due to numerical instability
> 4. Implement the spectral method for solving initial-value PDEs
> 5. Solve and interpret the results from simulations of the wave equation using the spectral method

## The FTCS method

In the previous lecture, we were looking at solving for the steady-state solution of a PDE such as the Poisson equation subjected to particular boundary conditions.
In that case, we were interested in a function $\phi(x,y)$ that was static in time but depended on two spatial coordinates. 

Another class of PDE that arises frequently in physics is a multivariable problem where one is given an initial condition for a function, and one would like to solve for the value of the function at later times.
Since time is now a variable in the equation, the simplest of these involve a function of one spatial dimension and time.
An example of this type of PDE is the diffusion equation.
For a general function $\phi(x,t)$, this has the form:

$$
\frac{\partial \phi}{\partial t} = 
D 
\frac{\partial^2 \phi}{\partial x^2} 
$$

One approximation for solving this is to use the equation above to make a prediction for $\phi(x,t+dt)$:

$$
\phi(x, t+dt) \approx \phi(x,t) + \frac{\partial \phi}{\partial t}dt  = \phi(x,t) +
D  \frac{\partial^2 \phi}{\partial x^2} dt
$$

Using the finite-difference approximation we used previously for the second derivative, we have:

$$
\phi(x, t+dt) \approx \phi(x,t) + \frac{D \cdot dt}{(dx)^2} \big[ \phi(x+dx) + \phi(x-dx) - 2 \phi(x) \big]
$$

This equation then allows us to calculate $\phi(x,t)$ by taking incremental steps in time and iteratively calculating the new updated $\phi$ at each time step.
This technique is known as the forward-time centered-space method (FTCS). 

Below, we will explore implementing the FTCS method for calculating the time dependence of the temperature of a block that is subjected to boundary conditions of fixed temperature, illustrated here schematically:

![](https://raw.githubusercontent.com/curiousbeams/lecture-tn23015-computational-science/main/figures/thermal_gradient.png)

### Exercise 11.1

A steel block that is 1 cm thick has an initial temperature of 20 degrees Celsius.
At time $t=0$, it is sandwiched between a cold bath at a temperature of 0 degrees at $x = 1$ cm, and a hot bath at a temperature of 50 degrees at $x = 0$.
Calculate the time dependence of the temperature profile inside the steel block. 

The thermal diffusivity D for steel is $4.25 \times 10^{-6}$ m$^2$/s.
Perform your computation for a grid of points in spatial coordinate $x$ that runs from 0 to 1 cm with a total of `Nx` = 100 points in space.
Calculate the profile for times from $t=0$ to $t=2$ seconds with `Nt` = 10$^4$ points in time.

In [ ]:
# Fill these in as you work through the exercise.
dt = dx = t = t1 = t2 = x = None

# Physical constants
D = 4.25e-6

# Our spatial axis
L = 0.01 
Nx = 100

# x = np.linspace(...)
# dx = ...

# Our time axis
tf = 2
Nt = int(1e4)

# t = np.linspace(...,...,...)
# dt = t[...]

# A matrix to store the time and spatial dependence
# of the temperature.
# This will be a matrix of size (N_t, N_x). 
# T[i,j] is then T for time t[i] and position x[j]
T = np.zeros([Nt, Nx])

# Our initial condibtion (uniform constant temperature)
# T[...] = 20

# The temperature at x=0 should be fixed at 50 for all times. 
# At x=L, the temperature should be fixed at 0 for all times. 
# T[...] = 50
# T[...] = 0

# t1 = time()
# for i in range(N_t-1):
#     # Note that we don't update the boundaries at x=0 and x=L
#     for j in range(...,...):
#         T[i+1,j] = T[i,j] + ... 
# t2 = time()

# Do not edit or remove the boiler-plate code below.
if given(dt, dx, t, t1, t2, x):
    print("Total calculation time: %.2f seconds" % (t2-t1))

show(dt=dt, dx=dx, t=t, t1=t1, t2=t2, x=x)

In [ ]:
check_answers(T=T, key="answer_14_1", when=given(dt, dx, t, t1, t2, x))

Let's take a look at the results.
A nice way to explore this is to step through time and watch
the temperature profile change.
The slider below chooses which time step to draw; because the
cells are reactive, the plot redraws itself as you move it.

In [ ]:
# We have 10^4 time steps, but 100 of them is plenty to see what happens.
frames = 100
step = slider(0, frames - 1, value=0, label="Time step", show_value=True)

md("*The slider is shown with the plot below.*")

In [ ]:
# Do not edit or remove the boiler-plate code below.
_ax = None
if given(t, x):
    _i = step.value * (Nt // frames)
    _fig, _ax = plt.subplots()
    _ax.plot(x, T[_i, :])
    _ax.set_ylim(0, 55)
    _ax.set_ylabel("T (deg C)")
    _ax.set_xlabel("Distance x (m)")
    _ax.set_title("t = %.2f seconds" % t[_i])

show(step, _ax, t=t, x=x)

## Applying the FTCS method to the wave equation

Another common initial-value problem that arises with partial differential equations is the wave equation.
An example of a common wave equation is that for the vertical displacements $u$ of a string as a function of the horizontal position $x$ and time $t$: 

![](https://raw.githubusercontent.com/curiousbeams/lecture-tn23015-computational-science/main/figures/string.png)

which obey the equation: 

$$
\frac{\partial^2 u}{\partial t^2} = c^2 
\frac{\partial^2 u}{\partial x^2}
$$

where $c$ is the wave velocity of the string.
How do we solve this equation?
Like we did for ODEs, we will split the second order derivative in time into two first order derivatives by introducing a new variable:

$$
v(x,t) \equiv \frac{\partial u}{\partial t}
$$

Physically, $v$ represents the vertical velocity of the vibrating string.
Doing so, we can split the equation above that has a second-order derivative in time into two, coupled equations that have only first-order derivatives in time:

$$
\begin{aligned}
\frac{\partial u}{\partial t} & = v \\
\frac{\partial v}{\partial t} & = c^2 \frac{\partial^2 u}{\partial x^2} \\
\end{aligned}
$$

Using Euler's method for this, and using the finite difference approximation for the second derivative with respect to $x$, we can derive the following two update equations for $u$ and $v$: 

$$
\begin{aligned}
u(x,t+dt) & = u(x,t) + v(x,t) dt \\
v(x,t+dt) & = v(x,t) + \frac{c^2 \cdot dt}{(dx)^2} [u(x+dx,t) + u(x-dx,t) - 2u(x,t)]
\end{aligned}
$$

which, in principle, we can solve using the FTCS method as we did above for the diffusion equation.

### Exercise 11.2

Fill in the code below to solve the wave equation for a vibrating string using the FTCS method.
Take $c = 1$ m/s and consider a string of length $L$. For your $x$ coordinate, use a grid with `Nx` = 100 points. 

For your $t$ coordinate, run your simulation for 1.2 seconds with 550 points in your time array.
Consider an initial displacement $u(x,t=0) = \sin(\pi x/L)$ and an initial velocity $v(x,t=0) = 0$.

In [ ]:
# Fill these in as you work through the exercise.
t_2 = u = v = x_2 = None

# The wave velocity in the string
c = 1 # m/s

# Positions along the length of the string
# Let's make it 1 m long
L_2 = 1
Nx_2 = 100
# x_2 = np.linspace(...)

# t_2 = np.linspace(...)
# u = np.zeros(...)
# v = np.zeros(...)

# Do not edit or remove the boiler-plate code below.
Nt_2 = dt_2 = dx_2 = t1_2 = t2_2 = tf_2 = None
if given(t_2, u, v, x_2):
    dx_2 = x_2[1]

    # Our time axis
    # Let's make it 1.2 seconds long
    Nt_2 = 550
    tf_2 = 1.2
    dt_2 = t_2[1]

    # A matrix for tracking the vertical displacement of the string
    # and the vertical velocity v = du/dt

    # Our initial condibtion: a "displaced" string
    # Sine wave in displacement with no initial velocity
    u[0,:] = np.sin(np.pi*x_2/L_2)
    v[0,:] = 0

    # Make sure the boundary conditions are enforced
    u[:,[0,-1]] = 0
    v[:,[0,-1]] = 0

    t1_2 = time()
    for _i in range(Nt_2-1):
        for _j in range(1,Nx_2-1):
            ...

            #u[i+1,j] = ...
            #v[i+1,j] = ...

    t2_2 = time()

    print("Total calculation time: %.2f seconds" % (t2_2-t1_2))

show(t=t_2, u=u, v=v, x=x_2)

In [ ]:
check_answers(u=u, v=v, key="answer_14_2")

Let's step through your results:

In [ ]:
frames_2 = 100
step_2 = slider(0, frames_2 - 1, value=0, label="Time step", show_value=True)

md("*The slider is shown with the plot below.*")

In [ ]:
# Do not edit or remove the boiler-plate code below.
_ax = None
if given(Nt_2, t_2, u, x_2):
    _i = step_2.value * (Nt_2 // frames_2)
    _fig, _ax = plt.subplots()
    _ax.plot(x_2, u[_i, :])
    _ax.set_ylim(-1.2, 1.2)
    _ax.set_ylabel("Displacement u (m)")
    _ax.set_xlabel("Distance x (m)")
    _ax.set_title("t = %.4f seconds" % t_2[_i])

show(step_2, _ax, Nt=Nt_2, t=t_2, u=u, x=x_2)

Clearly, the FTCS method does not work so well for the wave equation!

Why not?
Any profile on the grid can be written as a sum of Fourier modes, and one FTCS step multiplies each mode by an amplification factor $G$.
If $|G| > 1$ for even a single mode, that mode grows exponentially, starting from nothing more than round-off error.
For the diffusion equation every mode is damped as long as $r = D\,\Delta t/\Delta x^2 \leq 1/2$, which is why your steel block behaved; above that, the zig-zag mode takes over.
For the wave equation, stepping forward in time like Euler's method gives $|G| > 1$ for *every* mode, whatever the timestep.
Try both below.

In [ ]:
# An interactive figure from the book: run this cell to draw it. It loads from the internet.
from tn23015 import interactive_figure

await interactive_figure(
    "https://curiousbeams.github.io/numerics-widgets/notebooks/ftcs-stability.html",
    cells=["controlsView", "panelsView", "readoutView"],
    # theme="dark",  # if the colours look wrong, uncomment this (or use "light")
)

## Spectral Method

A very efficient way to solve the wave equation above is one that you are already (or should be) *very* familiar with from quantum mechanics from solving the Schroedinger equation!
(although you may not know its name)

This method is called the "spectral method".
The idea is that the linear wave equation above can be solved using separation of variables, which leads to two separate equations, one in time and the other in space, which you can solve separately.

The solutions of the spatial equation that results from the separation of variables gives you the shapes of the [normal modes](https://en.wikipedia.org/wiki/Normal_mode), or in quantum mechanics, the eigenmodes. 

The behaviour of eigenmodes in time is very simple: they always have the same shape but they oscillate in time.
If your initial condition is an eigenmode, then your time dependence is very simple!

To find the time dependent behaviour of **any** initial condition, the idea is to write the initial position and velocity as a superposition of the normal modes.
To get the solution at any time in the future, you can keep the same coefficients but add the specific sinusoidal oscillations for each mode!

In general, to find the eigenmodes, you have to first solve the spatial part of the equation: in quantum mechanics, this is what you spend a lot of time doing for different examples of $V(x)$. 

We will consider again the example of our vibrating string, fixed at two ends.
In this case, the eigenmodes are defined by the following equation:

$$
c^2 
\frac{\partial^2 u}{\partial x^2} = \lambda u
$$

This equation (the same as that for the wave function of the infinite square well) has solutions we know: these are sine functions!
Thus, we need to write our initial displacement, and initial velocity, as a superposition of sine functions (see also Newman section 9.3.4 for a good explanation), and then we can find $u(x,t)$ for all times!
If $u_k(0)$ and $v_k(0)$ are the Fourier coefficients at $t = 0$, then Fourier coefficient $u_k(t)$ at a time $t$ later is given by:

$$
u_k(t) = u_k(0) \cos(\omega_k t) + \frac{v_k(0)}{\omega_k} \sin(\omega_k t)
$$

with 

$$
\omega_k = \frac{\pi c k}{L}
$$

and $k$ is the index of the Fourier sine coefficient (starting from 1).
We can then find $u(x,t)$ by just taking the inverse Fourier sine transform of $u_k(t)$.

The decomposition of a function into sine waves is a common task (it is actually a Discrete Sine Transform (DST), a type of Fourier transform!).
Fortunately, there are pre-programmed functions built into the `scipy` package for the DST, and for the inverse DST, which can help save us the time of coding them ourselves: [`scipy.fftpack.dst`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.fftpack.dst.html).

One confusing aspect of the DST is that there are many different "types", which have slightly different conventions of the offsets and endpoints of the sine waves used in the transform.
The one we will use is "type I".

One peculiarity of the routines is that none of them include the "fixed" endpoints of our string at $x=0$ and $x=L$: since we want to have them in our function, we will have to take them off for performing the transform, and put them back on when we are done.
Another confusing thing is that the inverse transform when using "type I" is not normalized.
As all of these are quite inconvenient, we will build some "wrapper functions" around these to fix these for our purposes.

In [ ]:
# Notebook code

def my_dst(u):
    return dst(u[1:-1], type=1)

def my_idst(uk):
    u = np.zeros(len(uk)+2)
    u[1:-1] = idst(uk, type=1)/2/(len(uk)+1)
    return u

show()

### Exercise 11.3

Implement the spectral method to find $u(x,t)$ for the problem of Exercise 11.2. At the end of your code, implement a function that calculates $u(x,t)$, and use this to find $u(x,10)$.

In [ ]:
# Fill these in as you work through the exercise.
k = uk = uk0 = vk0 = w = None

# The wave velocity in the string
c_2 = 1 # m/s

# Positions along the length of the string
# Let's make it 1 m long
L_3 = 1
Nx_3 = 100
x_3 = np.linspace(0,L_3,Nx_3)
dx_3 = x_3[1]

# Our initial condibtion. u = vertical displacement, v = du/dt
u0 = np.sin(np.pi*x_3/L_3)
v0 = np.zeros(Nx_3)

# Calculate the Fourier coefficients of the initial condition

# uk0 = my_dst(...)
# vk0 = my_dst(...)

# k = ...
# w = ...

def calc_u(t):
    #uk = ...
    return my_idst(uk)

# Do not edit or remove the boiler-plate code below.
# `calc_u` raises on an unfilled `uk` rather than returning something the checker can recognise
# as unanswered, so unlike the other answers it has to be gated.
u_at_10 = calc_u(10) if given(uk) else None

show(k=k, uk=uk, uk0=uk0, vk0=vk0, w=w)

In [ ]:
check_answers(
    uk0=uk0,
    vk0=vk0,
    k=k,
    w=w,
    u_at_10=u_at_10,
    key="answer_14_3",
)

We can also step through it in time:

In [ ]:
frames_3 = 100
t_3 = np.linspace(0, 2, frames_3)
step_3 = slider(0, frames_3 - 1, value=0, label="Time step", show_value=True)

md("*The slider is shown with the plot below.*")

In [ ]:
# Do not edit or remove the boiler-plate code below.
_ax = None
if given(uk):
    _fig, _ax = plt.subplots()
    _ax.plot(x_3, calc_u(t_3[step_3.value]))
    _ax.set_ylim(-1.2, 1.2)
    _ax.set_ylabel("Displacement u (m)")
    _ax.set_xlabel("Distance x (m)")
    _ax.set_title("t = %.4f seconds" % t_3[step_3.value])

show(step_3, _ax, uk=uk)

Finally, this is a fun initial condition!

In [ ]:
u0_2 = np.exp(-(x_3-0.2)**2*500)
v0_2 = np.zeros(Nx_3)

uk0_2 = my_dst(u0_2)
vk0_2 = my_dst(v0_2)

# `calc_u` above is tied to the initial condition of Exercise 11.3, and marimo gives each cell
# its own `uk0`, so this initial condition needs its own version of the same one-liner.
def calc_u_2(t):
    return my_idst(uk0_2*np.cos(w*t) + vk0_2/w*np.sin(w*t))

show(u0=u0_2)

In [ ]:
frames_4 = 100
t_4 = np.linspace(0, 2, frames_4)
step_4 = slider(0, frames_4 - 1, value=0, label="Time step", show_value=True)

md("*The slider is shown with the plot below.*")

In [ ]:
# Do not edit or remove the boiler-plate code below.
_ax = None
if given(w):
    _fig, _ax = plt.subplots()
    _ax.plot(x_3, calc_u_2(t_4[step_4.value]))
    _ax.set_ylim(-1.2, 1.2)
    _ax.set_ylabel("Displacement u (m)")
    _ax.set_xlabel("Distance x (m)")
    _ax.set_title("t = %.4f seconds" % t_4[step_4.value])

show(step_4, _ax, w=w)